# Silver Layer Cleaning and Validation

This notebook cleans the Bronze Layer CSV files and produces Silver Layer outputs for the FinMark data pipeline.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
SILVER_DIR = BASE_DIR / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

## Cleaning rules

- Drop undocumented `col_*` junk columns.
- Remove exact duplicates.
- Convert dates/timestamps and numeric columns to proper types.
- Validate required fields.
- Flag data quality issues instead of deleting business-important rows.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
SILVER_DIR = BASE_DIR / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

validation_rows = []
issue_rows = []

def col_junk(df):
    return [c for c in df.columns if c.startswith("col_")]

def add_validation(file, check, status, details):
    validation_rows.append({"file": file, "check": check, "status": status, "details": details})

def add_issue(file, issue_type, count, handling):
    issue_rows.append({"file": file, "issue_type": issue_type, "count": int(count), "handling": handling})

def clean_event_logs():
    file = "event_logs.csv"
    df = pd.read_csv(RAW_DIR / file)
    junk = col_junk(df)
    df = df.drop(columns=junk)
    add_validation(file, "Dropped undocumented junk columns", "PASS", f"Dropped {len(junk)} columns")

    before = len(df)
    df = df.drop_duplicates().copy()
    add_issue(file, "Exact duplicate rows", before - len(df), "Removed before Silver export")

    for c in ["user_id", "event_type", "product_id"]:
        df[c] = df[c].astype(str).str.strip()
    df["user_id"] = df["user_id"].str.upper()
    df["product_id"] = df["product_id"].str.upper()
    df["event_type"] = df["event_type"].str.lower()

    df["event_timestamp"] = pd.to_datetime(df["event_time"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["event_date"] = df["event_timestamp"].dt.date.astype("string")
    df["event_hour"] = df["event_timestamp"].dt.hour.astype("Int64")
    df["is_checkout"] = df["event_type"].eq("checkout")
    df["is_amount_missing"] = df["amount"].isna()
    df["amount_quality_status"] = np.select(
        [df["is_checkout"] & df["amount"].isna(),
         df["is_checkout"] & df["amount"].notna() & (df["amount"] > 0),
         ~df["is_checkout"] & df["amount"].notna(),
         ~df["is_checkout"] & df["amount"].isna()],
        ["missing_checkout_amount", "valid_checkout_amount", "non_checkout_amount_present", "not_applicable"],
        default="review_required"
    )

    valid_events = {"login", "logout", "checkout", "wishlist_add", "profile_update", "page_view", "search", "add_to_cart"}
    invalid_event_count = (~df["event_type"].isin(valid_events)).sum()
    add_validation(file, "Allowed event_type values", "PASS" if invalid_event_count == 0 else "FAIL", f"{invalid_event_count} invalid values")

    missing_checkout = ((df["event_type"] == "checkout") & df["amount"].isna()).sum()
    non_checkout_amt = ((df["event_type"] != "checkout") & df["amount"].notna()).sum()
    add_issue(file, "Checkout rows with missing amount", missing_checkout, "Kept rows and flagged")
    add_issue(file, "Non-checkout rows with amount present", non_checkout_amt, "Kept rows and flagged")

    df = df[["user_id", "event_type", "event_timestamp", "event_date", "event_hour", "product_id", "amount", "is_checkout", "is_amount_missing", "amount_quality_status"]]
    df.to_csv(SILVER_DIR / "event_logs_clean.csv", index=False)
    return df

def clean_marketing_summary():
    file = "marketing_summary.csv"
    df = pd.read_csv(RAW_DIR / file)
    junk = col_junk(df)
    df = df.drop(columns=junk)
    add_validation(file, "Dropped undocumented junk columns", "PASS", f"Dropped {len(junk)} columns")

    before = len(df)
    df = df.drop_duplicates().copy()
    add_issue(file, "Exact duplicate rows", before - len(df), "Removed before Silver export")

    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")
    df["users_active"] = pd.to_numeric(df["users_active"], errors="coerce").astype("Int64")
    df["total_sales"] = pd.to_numeric(df["total_sales"], errors="coerce")
    df["new_customers"] = pd.to_numeric(df["new_customers"], errors="coerce").astype("Int64")
    df["report_generated"] = pd.to_datetime(df["report_generated"], errors="coerce")
    df["report_hour"] = df["report_generated"].dt.hour.astype("Int64")
    df["is_1600_batch_report"] = df["report_hour"].eq(16)

    for c in ["date", "users_active", "total_sales", "new_customers", "report_generated"]:
        n = df[c].isna().sum()
        add_validation(file, f"Required field: {c}", "PASS" if n == 0 else "FAIL", f"{n} missing/invalid values")

    df = df[["date", "users_active", "total_sales", "new_customers", "report_generated", "report_hour", "is_1600_batch_report"]]
    df.to_csv(SILVER_DIR / "marketing_summary_clean.csv", index=False)
    return df

def clean_trend_report():
    file = "trend_report.csv"
    df = pd.read_csv(RAW_DIR / file)
    junk = col_junk(df)
    df = df.drop(columns=junk)
    add_validation(file, "Dropped undocumented junk columns", "PASS", f"Dropped {len(junk)} columns")

    before = len(df)
    df = df.drop_duplicates().copy()
    add_issue(file, "Exact duplicate rows", before - len(df), "Removed before Silver export")

    df["week"] = df["week"].astype(str).str.strip()
    df["avg_users"] = pd.to_numeric(df["avg_users"], errors="coerce").astype("Int64")
    df["sales_growth_rate"] = pd.to_numeric(df["sales_growth_rate"], errors="coerce")
    df["week_start_date"] = pd.to_datetime(df["week"] + "-1", format="%G-W%V-%u", errors="coerce").dt.date.astype("string")

    for c in ["week", "week_start_date", "avg_users", "sales_growth_rate"]:
        n = df[c].isna().sum()
        add_validation(file, f"Required field: {c}", "PASS" if n == 0 else "FAIL", f"{n} missing/invalid values")

    df = df[["week", "week_start_date", "avg_users", "sales_growth_rate"]]
    df.to_csv(SILVER_DIR / "trend_report_clean.csv", index=False)
    return df

if __name__ == "__main__":
    clean_event_logs()
    clean_marketing_summary()
    clean_trend_report()
    pd.DataFrame(validation_rows).to_csv(SILVER_DIR / "silver_validation_report.csv", index=False)
    pd.DataFrame(issue_rows).to_csv(SILVER_DIR / "silver_quality_issues.csv", index=False)
    print("Silver Layer cleaning completed. Outputs saved to data/silver/.")


Silver Layer cleaning completed. Outputs saved to data/silver/.


In [7]:
events = pd.read_csv(SILVER_DIR / "event_logs_clean.csv")
marketing = pd.read_csv(SILVER_DIR / "marketing_summary_clean.csv")
trend = pd.read_csv(SILVER_DIR / "trend_report_clean.csv")

print("event_logs_clean:", events.shape)
print("marketing_summary_clean:", marketing.shape)
print("trend_report_clean:", trend.shape)

display(events.head())
display(marketing.head())
display(trend.head())

event_logs_clean: (2000, 10)
marketing_summary_clean: (100, 7)
trend_report_clean: (20, 4)


,user_id,event_type,event_timestamp,event_date,event_hour,product_id,amount,is_checkout,is_amount_missing,amount_quality_status
0,U0099,checkout,2023-06-03 04:13:00,2023-06-03,4,P010,NaN,True,True,missing_checkout_amount
1,U0240,wishlist_add,2023-06-03 05:08:00,2023-06-03,5,P020,2900.63,False,False,non_checkout_amount_present
2,U0374,profile_update,2023-06-05 06:22:00,2023-06-05,6,P028,NaN,False,True,not_applicable
3,U0122,page_view,2023-06-06 03:45:00,2023-06-06,3,P001,NaN,False,True,not_applicable
4,U0211,wishlist_add,2023-06-03 12:38:00,2023-06-03,12,P015,1728.27,False,False,non_checkout_amount_present


,date,users_active,total_sales,new_customers,report_generated,report_hour,is_1600_batch_report
0,2023-06-01,179,81287.31,9,2023-06-01 16:00:00,16,True
1,2023-06-02,67,74771.99,5,2023-06-02 16:00:00,16,True
2,2023-06-03,369,84809.74,11,2023-06-03 16:00:00,16,True
3,2023-06-04,94,61212.30,3,2023-06-04 16:00:00,16,True
4,2023-06-05,402,80911.49,10,2023-06-05 16:00:00,16,True


,week,week_start_date,avg_users,sales_growth_rate
0,2023-W21,2023-05-22,328,-0.003
1,2023-W22,2023-05-29,280,0.088
2,2023-W23,2023-06-05,130,0.073
3,2023-W24,2023-06-12,291,0.077
4,2023-W25,2023-06-19,200,-0.003


In [8]:
validation = pd.read_csv(SILVER_DIR / "silver_validation_report.csv")
issues = pd.read_csv(SILVER_DIR / "silver_quality_issues.csv")
display(validation)
display(issues)

,file,check,status,details
0,event_logs.csv,Dropped undocumented junk columns,PASS,Dropped 45 columns
1,event_logs.csv,Allowed event_type values,PASS,0 invalid values
2,marketing_summary.csv,Dropped undocumented junk columns,PASS,Dropped 45 columns
3,marketing_summary.csv,Required field: date,PASS,0 missing/invalid values
4,marketing_summary.csv,Required field: users_active,PASS,0 missing/invalid values
5,marketing_summary.csv,Required field: total_sales,PASS,0 missing/invalid values
6,marketing_summary.csv,Required field: new_customers,PASS,0 missing/invalid values
7,marketing_summary.csv,Required field: report_generated,PASS,0 missing/invalid values
8,trend_report.csv,Dropped undocumented junk columns,PASS,Dropped 47 columns
9,trend_report.csv,Required field: week,PASS,0 missing/invalid values


,file,issue_type,count,handling
0,event_logs.csv,Exact duplicate rows,0,Removed before Silver export
1,event_logs.csv,Checkout rows with missing amount,142,Kept rows and flagged
2,event_logs.csv,Non-checkout rows with amount present,866,Kept rows and flagged
3,marketing_summary.csv,Exact duplicate rows,0,Removed before Silver export
4,trend_report.csv,Exact duplicate rows,0,Removed before Silver export
